In [16]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd
import altair as alt


In [17]:
old_df = pd.read_parquet("../data/processed/merged.parquet")

In [18]:
df = pd.read_parquet("../data/processed/processed.parquet")

In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   parent_asin                    20000 non-null  str    
 1   product_title                  20000 non-null  str    
 2   features                       20000 non-null  object 
 3   description                    20000 non-null  object 
 4   categories                     20000 non-null  object 
 5   details                        20000 non-null  object 
 6   price                          11212 non-null  float64
 7   derived_avg_rating             8669 non-null   float64
 8   max_helpful_vote               8669 non-null   float64
 9   n_reviews                      20000 non-null  int64  
 10  review_docs                    8669 non-null   object 
 11  review_text                    8669 non-null   str    
 12  candidate_review_title         8669 non-null   str    
 1

In [5]:
df.head()

,parent_asin,product_title,features,description,categories,details,price,derived_avg_rating,max_helpful_vote,n_reviews,review_docs,review_text,candidate_review_title,candidate_review_text,candidate_review_helpful_vote
0,B09W5PMK5X,COMFEE’ Washing Machine 2.4 Cu.ft LED Portable...,[AUTO CLEAN - COMFEE' 2.4 Cu.ft portable washe...,[Feeling unsafe going to Laundromat? Tired of ...,"[Appliances, Laundry Appliances, Washers & Dry...","[(Brand Name, ""COMFEE'""), (Model Info, ""CLV24N...",399.00,4.522727,11,44,[so far so good: but i havent had it long a y...,so far so good: but i havent had it long a ye...,Would be great but it's defective: keeps drain...,"As a number of other reviews have noticed, the...",11
1,B092T5Y46X,Nispira Activated Charcoal Water Filters Repla...,[Refined coconut shell activated charcoal wate...,[Refined coconut shell activated charcoal wate...,"[Small Appliance Parts & Accessories, Coffee &...","[(Product Dimensions, ""7.1 x 6.2 x 2.4 inches""...",16.99,4.684783,2,92,[great price: I ordered these after looking in...,great price: I ordered these after looking int...,great price,I ordered these after looking into replacement...,2
2,B08CXQLP21,BIMZUC Christmas Refrigerator Door Handle Cove...,[🎅【High Quality Fabric】Handle covers set are s...,[],"[Appliances, Parts & Accessories, Refrigerator...","[(Manufacturer, ""YoGus""), (Part Number, ""Handl...",NaN,4.666667,1,3,[Festive!: These door handle covers are quite ...,Festive!: These door handle covers are quite f...,Festive!,These door handle covers are quite festive. It...,1
3,B0BZW5SSVK,Electactic Ice Maker Countertop Portable Ice M...,[[Light but Efficient] This ice maker is the l...,"[Electactic Portable Ice Maker Machine, This i...","[Appliances, Refrigerators, Freezers & Ice Mak...","[(Brand, ""Electactic""), (Product Dimensions, ""...",84.99,3.875000,34,8,"[Easy too use,not very fast: I like the ice cu...","Easy too use,not very fast: I like the ice cub...",Beware some claims,"First reason I bought it is the ""44 pounds in ...",34
4,B0B57GSDMD,AeroPress Replacement Filter Pack - Microfilte...,[Keep Pressing: Designed for use with AeroPres...,[],"[Small Appliance Parts & Accessories, Coffee &...","[(Product Dimensions, ""2.6 x 1.57 x 0.2 inches...",7.95,4.779817,29,109,[Good filters for the right coffeemaker: I ord...,Good filters for the right coffeemaker: I orde...,We love our aeropress and these work perfectly,works as advertised. We love our aeropress and...,29


In [6]:
missingness = pd.DataFrame(
    {"n_missing": df.isna().sum(), "prop_missing": df.isna().mean()}
).sort_values("prop_missing", ascending=False)

missingness

,n_missing,prop_missing
price,1519,0.318716
parent_asin,0,0.000000
product_title,0,0.000000
features,0,0.000000
description,0,0.000000
categories,0,0.000000
details,0,0.000000
derived_avg_rating,0,0.000000
max_helpful_vote,0,0.000000
n_reviews,0,0.000000


In [7]:
df.describe()

,price,derived_avg_rating,max_helpful_vote,n_reviews,candidate_review_helpful_vote
count,3247.000000,4766.000000,4766.000000,4766.000000,4766.000000
mean,71.156655,4.291282,3.206882,4.395930,3.206882
std,238.423778,1.121423,14.132309,14.202847,14.132309
min,1.940000,1.000000,0.000000,1.000000,0.000000
25%,13.990000,4.000000,0.000000,1.000000,0.000000
50%,23.960000,5.000000,0.000000,1.000000,0.000000
75%,44.995000,5.000000,2.000000,3.000000,2.000000
max,6999.000000,5.000000,425.000000,382.000000,425.000000
